In [12]:
import uuid
import re
import json
from pprint import pprint

import requests
from bs4 import BeautifulSoup

In [2]:
# получаем html код страницы, если она доступна, иначе Error
def get_html_code(url: str) -> str:
    headers = {
      'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        req = requests.get(url, headers=headers)
        # The HTTP 200 OK success status response code indicates that the request has succeeded.
        if req.status_code == 200:
            return req.text
        else:
            return "Error"
        
    except Exception as e:
        print(e)
        return "Error"

# переводим html код в объект soup класса BeautifulSoup
def html_to_soup(html: str) -> BeautifulSoup:
    soup = BeautifulSoup(html, "html.parser")
    return soup

# вытаскивает из текста совпадения, используя регулярные выражения
def find_by_regex(text: str, regular_expression: re.Pattern) -> list[str]:
    matches = re.findall(regular_expression, text)
    matches = list(set(matches))
    return matches

In [3]:
report_url = 'https://thedfirreport.com/2023/05/22/icedid-macro-ends-in-nokoyawa-ransomware'
html = get_html_code(report_url)
soup = html_to_soup(html)

In [4]:
domain_regex = r'\w+\[\.\][a-z]{1,6}'
ipv4_regex = r'\d{1,3}\.\d{1,3}\.\d{1,3}\[\.\]\d{1,3}'

domains = find_by_regex(soup.text, domain_regex)
ipv4 = find_by_regex(soup.text, ipv4_regex)

domains, ipv4

(['iconnectgs[.]com',
  'belliecow[.]wiki',
  'guaracheza[.]pics',
  'aicsoftware[.]com',
  'simipimi[.]com',
  'stayersa[.]art',
  'kicknocisd[.]com',
  'curabiebarristie[.]com',
  'dropmefiles[.]com'],
 ['23.29.115[.]152',
  '50.3.132[.]232',
  '137.74.104[.]108',
  '159.65.169[.]200',
  '5.8.18[.]242',
  '45.66.248[.]119'])

In [ ]:
"""Ключ необходимо запросить самостоятельно - https://inseca.tech/FAQ#Practic6"""
rst_threat_apikey = "здесь ваш апи ключ от rst THREAT feed"
assert rst_threat_apikey != "здесь ваш апи ключ от rst THREAT feed", f"Please provide a valid rst cloud api key."

from functools import lru_cache

@lru_cache
def ioc_lookup(ioc: str):

  if "[.]" in ioc:
    ioc = ioc.replace("[.]", ".")

  headers = {
      'accept': 'application/json',
      'x-api-key': rst_threat_apikey,
  }

  query_params = {
      'value': ioc,
  }

  try:
    response = requests.get('https://api.cyberthreattech.ru/v1/ioc', headers=headers, params=query_params)
    return response.json()
    
  except Exception as e:
    return e

In [20]:
response = ioc_lookup("45.66.248[.]119")
pprint(response, indent=2)

{ 'asn': { 'cloud': '',
           'domains': '5815',
           'firstip': {'netv4': '45.66.248.0', 'num': '759363584'},
           'isp': 'BVEUAS',
           'lastip': {'netv4': '45.66.249.255', 'num': '759364095'},
           'num': '62005',
           'org': ''},
  'collect': '1746489600',
  'cve': [],
  'description': 'IOC with tags: malware. Related threats: icedid',
  'fp': {'alarm': 'false', 'descr': ''},
  'fseen': '1665705600',
  'geo': {'city': 'Atlanta', 'country': 'United States', 'region': 'Georgia'},
  'id': '49582cbc-d900-3e76-8f46-4a302624e2d0',
  'industry': [],
  'ioc_type': 'ipv4',
  'ioc_value': '45.66.248.119',
  'lseen': '1746403200',
  'ports': ['-1', '443'],
  'related': {'domains': []},
  'score': { 'first': '60',
             'frequency': '0',
             'last': '60',
             'src': '67.49',
             'tags': '0.89',
             'total': '0'},
  'src': { 'name': ['github_repos', 'otx', 'threatreports'],
           'report': 'https://github.com/rod

In [21]:
description = response.get("description")
report_source = response.get("src", {}).get("report")
ioc_value = response.get("ioc_value")
ioc_type = response.get("ioc_type")
tag = description.split('.')[0].split(": ")[-1]
related_threat = description.split(".")[1].split(": ")[-1]

print("=== IOC Report ===")
print(f"Tag: {tag}")
print(f"Related Threat: {related_threat}")
print(f"IOC Type: {ioc_type}")
print(f"IOC Value: {ioc_value}")
print(f"Report Source: {report_source}")
print(f"Full Description: {description}")
print("==================")

=== IOC Report ===
Tag: malware
Related Threat: icedid
IOC Type: ipv4
IOC Value: 45.66.248.119
Report Source: https://github.com/rodanmaharjan/ThreatIntelligence/archive/refs/heads/main.zip,https://otx.alienvault.com,https://thedfirreport.com/2023/05/22/icedid-macro-ends-in-nokoyawa-ransomware
Full Description: IOC with tags: malware. Related threats: icedid


In [22]:
dict_data = {
  "ioc": ioc_value,
  "tag": tag,
  "related_threat": related_threat,
  "report_source": report_source,
  "ioc_type": ioc_type
}

In [23]:
def get_ioc_data(ioc: str) -> dict:
  #потом ставим try except
  try:
    response = ioc_lookup(ioc)
    report_source = response.get("src").get("report")
    tag = description.split(".")[0].split(": ")[-1]
    related_threat = description.split(".")[1].split(": ")[-1]
    ioc_type = response.get("ioc_type")

    dict_data = {
      "ioc": ioc,
      "tag": tag,
      "related_threat": related_threat,
      "report_source": report_source,
      "ioc_type": ioc_type
    }

    # return dict_data
  
  except Exception as e:
    dict_data = {
      "ioc": ioc,
      "tag": None,
      "related_threat": None,
      "report_source": None,
      "ioc_type": None
    }

  return dict_data

In [24]:
get_ioc_data("simipimi[.]com")

{'ioc': 'simipimi[.]com',
 'tag': 'malware',
 'related_threat': 'icedid',
 'report_source': 'https://github.com/rodanmaharjan/ThreatIntelligence/archive/refs/heads/main.zip,https://github.com/stamparm/maltrail/blob/master/trails/static/malware/icedid.txt',
 'ioc_type': 'domain'}

In [25]:
full_data = []

for domain in domains:
  data = get_ioc_data(domain)
  # print(data)
  full_data.append(data)

for ip in ipv4:
  data = get_ioc_data(ip)
  # print(data)
  full_data.append(data)

In [26]:
full_data

[{'ioc': 'iconnectgs[.]com',
  'tag': 'malware',
  'related_threat': 'icedid',
  'report_source': 'https://github.com/rodanmaharjan/ThreatIntelligence/archive/refs/heads/main.zip,https://github.com/stamparm/maltrail/blob/master/trails/static/malware/cobaltstrike-1.txt',
  'ioc_type': 'domain'},
 {'ioc': 'belliecow[.]wiki',
  'tag': 'malware',
  'related_threat': 'icedid',
  'report_source': 'https://github.com/rodanmaharjan/ThreatIntelligence/archive/refs/heads/main.zip,https://github.com/stamparm/maltrail/blob/master/trails/static/malware/icedid.txt',
  'ioc_type': 'domain'},
 {'ioc': 'guaracheza[.]pics',
  'tag': 'malware',
  'related_threat': 'icedid',
  'report_source': 'https://github.com/rodanmaharjan/ThreatIntelligence/archive/refs/heads/main.zip,https://github.com/stamparm/maltrail/blob/master/trails/static/malware/icedid.txt',
  'ioc_type': 'domain'},
 {'ioc': 'aicsoftware[.]com',
  'tag': 'malware',
  'related_threat': 'icedid',
  'report_source': 'https://github.com/rodanmah

In [68]:
def write_to_json(filename: str, data: list[dict]):
    with open(filename, 'w') as f:
        json.dump(data,
                  f,
                  indent=4,
                  ensure_ascii=True)

In [70]:
write_to_json("ioc.json", full_data)